# Calculating a Persistence Extreme Meteorological Year -- Methodology Walkthrough

Extreme meteorological years (XMYs) are intended to capture extreme events. We define two types of XMYs:

1. persistence: sustained extremes (ex: an unusually hot summer)
2. shock: system shocks, weather events (ex: heat waves, cold snaps)

This notebook covers to the steps to generate persistence XMYs.

The persistence XMY methodology utilizes the following changes to the TMY methodology to capture extremes:
1. Uses hourly air temperature, reflecting utility priorities when using XMYs.
2. Constructs the final XMY by selecting a year for each hour - rather than a year for each month - of the final representative year.
3. Selects each year using percentiles, rather than distance from a climatological CDF. This, essentially, applies the Cal-Adapt Standard Year methodology to persistence XMY construction.

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import hvplot.xarray
from importlib.resources import files
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt

import climakitae as ck
from climakitae.core.constants import UNSET
from climakitae.core.data_export import write_tmy_file

# Initialize ClimateData object
cd = ck.ClimateData(verbosity=-1)

### Step 1: Grab and process all required input data


#### Step 1a: Select location of interest
XMYs are calculated for a specific location of interest, like a building or power plant. Here, we will use a known weather station location, via their latitude and longitude to extract the data that we need to calculate the XMY. In the example below, we will look specifically at Los Angeles International Airport, but will note in the code below how you can provide your own location coordinates too.

In [2]:
# read in station file of CA HadISD stations
stn_filepath_s3 = "s3://cadcat/hadisd/hadisd_stations.csv"
stn_file = pd.read_csv(stn_filepath_s3, index_col=[0])
stn_file.head(5)

,state,station,city,ID,LAT_Y,LON_X,station id,elevation
0,CA,Bakersfield Meadows Field (KBFL),Bakersfield,KBFL,35.43424,-119.05524,72384023155,149.3
1,CA,Blythe Asos (KBLH),Blythe,KBLH,33.61876,-114.71451,74718823158,120.4
2,CA,Burbank-Glendale-Pasadena Airport (KBUR),Burbank,KBUR,34.19966,-118.36543,72288023152,222.7
3,CA,Needles Airport (KEED),Needles,KEED,34.76783,-114.61842,72380523179,270.6
4,CA,Fresno Yosemite International Airport (KFAT),Fresno,KFAT,36.77999,-119.72016,72389093193,101.9


In [3]:
# grab airport
stn_name = "Los Angeles International Airport (KLAX)"
stn_code = stn_file.loc[stn_file["station"] == stn_name]["station id"].item()
one_station = stn_file.loc[stn_file["station"] == stn_name]
stn_lat = one_station.LAT_Y.item()
stn_lon = one_station.LON_X.item()
stn_state = one_station.state.item()

# Output results
print("station coordinates:", (stn_lat, stn_lon))
print("station code:", stn_code)

station coordinates: (33.93816, -118.3866)
station code: 72295023174


Alternatively, you may want to provide your own location instead of one of the HadISD stations above. If so, uncomment the cell below by removing the `#` symbol and modifying the lines of code. Note, with custom locations, if an elevation is not provided, a default value of 0.0 m will be used. 

In [ ]:
## provide your own location, via latitude and longitude coordinates
# stn_lat = YOUR_LAT_HERE
# stn_lon = YOUR_LON_HERE
# stn_state = 'YOUR_STATE_HERE'
# stn_name = 'YOUR_STATION_NAME_HERE'
# stn_code = 'custom'
# stn_elev = YOUR_ELEV_HERE # meters

#### Step 1b: Select time frame of interest
The second required input for generating an XMY dataset is the **time frame of interest**. The recommended minimum number of input years for a TMY dataset is 15-20 years worth of daily data. For data post-2014, we will utilize SSP 3-7.0, although scenario selection in the near-future is relatively independent. If calculating a TMY for the far-future, **carefully consider which scenario SSP to include**, as there will be **significant** differences present. 

As an example, we will process the data for our designated station location (latitude, and longitude) at 3 km over the <span style="color:#FF0000">2005-2020</span> period in a time-based approach. 

> **Note:** We use a 15 year period here to keep the notebook runtime manageable, as this is a methods demonstration only. For production use, a 30-year period is recommended to represent a standard climatological baseline.

In [4]:
# selected reference period
start_year = 2005
end_year = 2020

#### Step 1c: Retrieve variables in catalog
We use 4 models from those available in Cal-Adapt to generate TMYs, which require solar variables. While XMY generation is not subject to the same variable constraints as TMYs, we will utilize those same 4 models for consistency. 

Lastly, because the dynamically downscaled WRF data in the Cal-Adapt: Analytics Engine is in UTC time, we'll convert to the timezone of the station we've selected. This is particularly important for the timing of solar radiation max and min, which is a critical piece of a TMY. The handy `convert_to_local_time` function corrects for this, and ensures that all data are within the same daily timestamp.

In [ ]:
# selected models
data_models = [
    "wrf_ucla_taiesm1_ssp370_r1i1p1f1",
    "wrf_ucla_ec-earth3_ssp370_r1i1p1f1",
    # "wrf_ucla_miroc6_ssp370_r1i1p1f1",
    # "wrf_ucla_mpi-esm1-2-hr_ssp370_r3i1p1f1",
]

# map variable names to descriptive names and units
var_info = {
    "t2": {"long_name": "Air temperature at 2m", "units": "degC"},
}

Now that we have set up the model list, let's retrieve our data! We have heard from utilities that air temperature is the single priority variable they need from XMYs. Instead of using all 10 TMY variables to identify which years are selected, then, we construct the persistence XMY based on hourly air temperature (`t2`) alone.

In [ ]:
t2_ds = (
    cd.catalog("cadcat")
    .variable("t2")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "convert_units": "degC",
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:
# Subset simulations and convert to DataArray
t2_ds = t2_ds.sel(sim=data_models, time=slice(str(start_year), str(end_year))).t2

# Load into memory using dask progress bar
with ProgressBar():
    t2_ds.load()

Now we'll remove the volcanic years: 

In [5]:
t2_ds = xr.load_dataset("hourly_air_temp_lax_2005_2020.nc")

In [6]:
# Remove the years for the Pinatubo eruption
t2_ds = t2_ds.where(~t2_ds['time'].dt.year.isin([1991, 1992, 1993, 1994]))

### Step 2: Select a Representative Year for Each Hour

In [20]:
data = t2_ds
q=0.9
skip_last=True

In [21]:
# Select air temperature
data = data['t2']

# Check for simulation dimension
has_simulation = "sim" in data.dims
if has_simulation:
    simulations = data.sim.values
else:
    simulations = [None]

if skip_last:  # Remove data from last month and year
    all_months = data["time"].dt.month.values
    last_month = int(all_months[-1])
    all_years = data["time"].dt.year.values
    last_year = int(all_years[-1])

    is_last_month = (data["time"].dt.year == last_year) & (
        data["time"].dt.month == last_month
    )
    data_test = data.where(~is_last_month)

In [24]:
# Get all available time data
hours_per_year = 8760
total_hours = len(data.time)
n_years = total_hours // hours_per_year

In [25]:
print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

# Create hour-of-year coordinate for all data (cycling through 1-8760)
hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
data = data.assign_coords(hour_of_year=("time", hour_of_year_all))

      📊 Processing 140,160 hours (16 years) of data
      🎯 Computing 90th percentile for each hour of year


In [26]:
# Initialize storage for profiles
df_list = []
for sim_idx, sim in enumerate(simulations):

    # Select data for this warming level and simulation combination
    if has_simulation:
        subset_data = data.isel(sim=sim_idx)
    else:
        subset_data = data
        
    # Vectorized quantile computation using numpy
    # Reshape raw values into (n_years, hours_per_year) then compute
    # the quantile across years for each hour-of-year position
    values = subset_data.values
    n_total = len(values)
    usable = (n_total // hours_per_year) * hours_per_year
    year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

    # Compute quantile targets for each of the 8760 hour positions
    quantile_targets = np.nanquantile(
        year_hour_matrix, q, axis=0
    )  # shape: (8760,)

    # For each hour position, find the actual year whose value is
    # closest to the quantile (avoids interpolation)
    diffs = np.abs(
        year_hour_matrix - quantile_targets[np.newaxis, :]
    )  # (n_years, 8760)

    closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)

    # Calendar year for each hour-of-year position
    usable_times = subset_data.time.values[:usable]
    row_years = pd.DatetimeIndex(usable_times).year.values.reshape(-1, hours_per_year)[:, 0]
    sel_year = row_years[closest_year_idx]  # (8760,) — calendar year, not row index

    # Store the selected years in a pandas dataframe
    df_i = pd.DataFrame({
        "hour": np.arange(1, hours_per_year + 1),
        "sim": sim,
        "year": sel_year.astype(int),
    })
    df_list.append(df_i)

# Concatenate list together for all simulations
top_df = pd.concat(df_list).reset_index(drop=True)

In [ ]:
def get_top_hours(data: xr.DataArray, q: float) -> pd.DataFrame:
    """
    Selects a representative year for each hour using the Cal-Adapt Standard Year
    methodology, using hourly air temperature.

    Computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns the year that value was taken from for each of the 8760 hours (one year), 
    for each simulation.

    Constructs a table of years or per hour per simulation for use in persistence XMY generation.

    Parameters
    ----------
    data : xr.DataArray
        Hourly air temperature with time and simulation dimensions. 

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Select air temperature
    data = data['t2']

    # Check for simulation dimension
    has_simulation = "sim" in data.dims
    if has_simulation:
        simulations = data.sim.values
    else:
        simulations = [None]

    if skip_last:  # Remove data from last month and year
        all_months = data["time"].dt.month.values
        last_month = int(all_months[-1])
        all_years = data["time"].dt.year.values
        last_year = int(all_years[-1])

        is_last_month = (data["time"].dt.year == last_year) & (
            data["time"].dt.month == last_month
        )
        data = data.where(~is_last_month)

    # Get all available time data
    hours_per_year = 8760
    total_hours = len(data.time)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time", hour_of_year_all))

    # Initialize storage for profiles
    df_list = []
    for sim_idx, sim in enumerate(simulations):

        # Select data for this warming level and simulation combination
        if has_simulation:
            subset_data = data.isel(sim=sim_idx)
        else:
            subset_data = data
            
        # Vectorized quantile computation using numpy
        # Reshape raw values into (n_years, hours_per_year) then compute
        # the quantile across years for each hour-of-year position
        values = subset_data.values
        n_total = len(values)
        usable = (n_total // hours_per_year) * hours_per_year
        year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

        # Compute quantile targets for each of the 8760 hour positions
        quantile_targets = np.nanquantile(
            year_hour_matrix, q, axis=0
        )  # shape: (8760,)

        # For each hour position, find the actual year whose value is
        # closest to the quantile (avoids interpolation)
        diffs = np.abs(
            year_hour_matrix - quantile_targets[np.newaxis, :]
        )  # (n_years, 8760)

        closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)

        # Calendar year for each hour-of-year position
        usable_times = subset_data.time.values[:usable]
        row_years = pd.DatetimeIndex(usable_times).year.values.reshape(-1, hours_per_year)[:, 0]
        sel_year = row_years[closest_year_idx]  # (8760,) — calendar year, not row index

        # Store the selected years in a pandas dataframe
        df_i = pd.DataFrame({
            "hour": np.arange(1, hours_per_year + 1),
            "sim": sim,
            "year": sel_year.astype(int),
        })
        df_list.append(df_i)

    # Concatenate list together for all simulations
    top_df = pd.concat(df_list).reset_index(drop=True)

    return top_df

In [ ]:
q=0.15
top_df = get_top_hours(t2_ds,q)

In [ ]:
top_df

### Step 3: Generate the Persistence XMY data outputs

Generally, the following data is outputted using the TMY months:
- Date & time (UTC)
- Air temperature at 2m [°C]
- Dew point temperature [°C]
- Relative humidity [%]
- Global horizontal irradiance [W/m2]
- Direct normal irradiance [W/m2]
- Diffuse horizontal irradiance [W/m2]
- Downwelling infrared radiation [W/m2]
- Wind speed at 10m [m/s]
- Wind direction at 10m [°]
- Surface air pressure [Pa]

The following function will retrieve all of this data for the designated TMY month and concatenate it together into a pandas dataframe so that we can export it as a csv file. We'll do this next. 

In [ ]:
def generate_persistence_xmy_data(
    top_df: pd.DataFrame,
    start_year: int,
    end_year: int,
    lat: float,
    lon: float,
    data_models: list,
) -> dict:
    """Generate persistence extreme meteorological year data.

    Parameters
    ----------
    top_df : pd.DataFrame
        Long-format frame with columns hour, simulation, and year. Each hour-sim-yr
        combo represents the top candidate that is closest to the desired percentile.
    start_year : int
    end_year : int
    lat : float
    lon : float
    data_models : list

    Returns
    -------
    dict of str : pd.DataFrame
        Dictionary in the format {simulation: XMY DataFrame}.
    """
    # Define climate data object with minimal terminal output
    cd = ck.ClimateData(verbosity=-2)

    # Define variables and unit conversions
    var_info = {
        "t2": {
            "long_name": "Air temperature at 2m (degC)",
            "units": "degC",
        },  # convert default K -> C
        "dew_point_2m": {
            "long_name": "Dew point temperature at 2m (degC)",
            "units": "degC",
        },  # convert default K -> C
        "relative_humidity_2m": {
            "long_name": "Relative humidity (0-100)",
            "units": UNSET,
        },  # Use unit default (0-100)
        "swdnb": {
            "long_name": "Instantaneous downwelling shortwave flux at bottom (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "swddni": {
            "long_name": "Shortwave surface downward direct normal irradiance (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "swddif": {
            "long_name": "Shortwave surface downward diffuse irradiance (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "lwdnb": {
            "long_name": "Instantaneous downwelling longwave flux at bottom (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "wind_speed_10m": {
            "long_name": "Wind speed at 10m (m/s)",
            "units": UNSET,
        },  # Use unit default (m/s)
        "wind_direction_10m": {
            "long_name": "Wind direction at 10m (degrees)",
            "units": UNSET,
        },  # Use unit default (degrees)
        "psfc": {
            "long_name": "Surface pressure (Pa)",
            "units": UNSET,
        },  # Use unit default (Pa)
    }

    ## ================== GET DATA FROM CATALOG ==================
    print(
        f"STEP 1: RETRIEVING HOURLY DATA FROM CATALOG FOR {len(var_info)} VARIABLES\n"
    )
    all_vars_list = []
    for i, (var, info) in enumerate(var_info.items(), start=1):
        long_name = info["long_name"]
        units = info["units"]
        print(f"({i}/{len(var_info)}) Variable: {var}")
        print("Retrieving data from catalog...")
        data_by_var = (
            cd.catalog("cadcat")
            .variable(var)
            .activity_id("WRF")
            .institution_id("UCLA")
            .table_id("1hr")
            .grid_label("d03")
            .experiment_id(["historical", "ssp370"])
            .processes(
                {
                    "time_slice": (start_year, end_year + 1),
                    "clip": (lat, lon),
                    "convert_units": units,
                    "convert_to_local_time": {"convert": "yes"},
                }
            )
            .get()
        )

        # Subset simulations and convert to DataArray
        data_by_var = data_by_var.sel(
            sim=data_models, time=slice(str(start_year), str(end_year))
        )[var]

        # Update variable name to use long_name for clarity
        data_by_var.name = long_name

        # print(f"Retrieved. Loading data into memory...")
        # with ProgressBar():
        #     data_by_var.load()

        print(f"Data loaded into memory.\n")
        all_vars_list.append(data_by_var)

    # Merge data from all variables into a single xr.Dataset object
    # Drop unneeded coordinates
    all_vars_ds = xr.merge(all_vars_list)
    all_vars_ds = all_vars_ds.drop_vars(
        ["lakemask", "landmask", "x", "y", "Lambert_Conformal"], errors="ignore"
    )

    # # Clear individual DataArrays from memory after merge
    # del all_vars_list

    ## ================== CONSTRUCT XMY (vectorized) ==================
    print("\nSTEP 2: CALCULATING PERSISTENCE EXTREME METEOROLOGICAL YEAR PER MODEL SIMULATION\n")
    HOURS_PER_YEAR = 8760

    # Trim to whole years and stamp hour-of-year / year coords ONCE
    n_total = all_vars_ds.sizes["time"]
    n_years = n_total // HOURS_PER_YEAR
    usable = n_years * HOURS_PER_YEAR
    ds = all_vars_ds.isel(time=slice(0, usable))

    hoy = np.tile(np.arange(1, HOURS_PER_YEAR + 1), n_years)
    yrs = pd.DatetimeIndex(ds.time.values).year.values
    ds = ds.assign_coords(hour_of_year=("time", hoy), year=("time", yrs))

    # Reshape time -> (year, hour_of_year). One xarray op for ALL variables.
    ds_2d = ds.set_index(time=["year", "hour_of_year"]).unstack("time")
    year_values = ds_2d.year.values  # ascending years available in the data

    xmy_df_all = {}

    def reformatted_time_coordinate(picked):
        step = pd.Timedelta(hours=1)
        years = picked.coords["year"].values
        hoy = picked.coords["hour_of_year"].values
        result = pd.to_datetime([f"{y}-01-01" for y in years]) + (hoy - 1) * step
        return result.strftime("%Y-%m-%d %H:%M")

    for sim in ds_2d.sim.values:
        print(f"Calculating persistence XMY for simulation: {sim}")

        # Vectorized lookup of selected year per hour-of-year (replaces L137–139)
        sel_years = (
            top_df[top_df["sim"] == sim]
            .sort_values("hour")["year"]
            .to_numpy()
        )  # shape (8760,)
        year_idx = np.searchsorted(year_values, sel_years)

        # Single fancy-index gather (replaces the 8760-iteration loop body)
        hour_da = xr.DataArray(np.arange(HOURS_PER_YEAR), dims="hour_of_year")
        yidx_da = xr.DataArray(year_idx, dims="hour_of_year")
        picked = ds_2d.sel(sim=sim).isel(year=yidx_da, hour_of_year=hour_da)

        df = picked.to_dataframe().reset_index()
        df["time"] = reformatted_time_coordinate(picked)
        df = df.drop(columns=['hour_of_year','year'])
        xmy_df_all[sim] = df
    
        print("PERSISTENCE XMY PROCESS COMPLETE.")

    return xmy_df_all

In the next cell, we will run the `generate_persistence_xmy_data` function which will retrieve, subset, and format the data for each month according to the TMY months for all requested variables. We have included print statements so you can watch the progress for each variable in each month as it builds. 

<span style="color:#FF0000">**Note**: <span style="color:#000000"> This will take time! On the Analytics Engine JupyterHub, this takes approximately 22 minutes. Progress bars will indicate the status of generating the TMY data for each simulation. 

In [ ]:
xmy_data_to_export = generate_persistence_xmy_data(
    top_df, start_year, end_year, stn_lat, stn_lon, data_models
)

Let's observe what the TMY data looks like for one of the simulations:

In [ ]:
simulation = "wrf_ucla_miroc6_ssp370_r1i1p1f1"
xmy_data_to_export[simulation].head(5)

Next, we visualize the TMY data itself.

In [ ]:
# Make plot using pandas
ax = xmy_data_to_export[simulation].plot(
    x="time",
    y=[
        col
        for col in xmy_data_to_export[simulation]
        if col not in ["time", "sim", "lat", "lon"]
    ],
    subplots=True,
    figsize=(12, 12),
    legend=True,
    fontsize=10,
)

# Plot formatting
for a in ax.flatten():
    a.xaxis.label.set_fontsize(12)
    a.yaxis.label.set_fontsize(12)
    a.legend(loc="upper right", fontsize=14)
plt.suptitle(
    f"Persistence Exreme Meteorological Year (simulation: {simulation})", fontsize=16, y=0.98
)
plt.tight_layout();

Lastly, let's export the XMY data below as csv files. There will be a file per simulation downloaded. When utilizing XMY data in your own workflows, we recommend that **all simulations are considered** in your analyses, especially for future scenarios. Note, if you are working with a custom location, please also provide the optional `stn_elev` argument to `write_tmy_file`; if no elevation value is provided, an elevation value of 0.0 is set as the default. 

In [ ]:
p = q * 100
p = int(p)
for sim, tmy in xmy_data_to_export.items():
    filename = "persistence_XMY_p{0}_{1}_{2}".format(
        p, stn_name.replace(" ", "_").replace("(", "").replace(")", ""), sim
    ).lower()
    write_tmy_file(
        filename,
        xmy_data_to_export[sim],
        (start_year, end_year),
        stn_name,
        stn_code,
        stn_lat,
        stn_lon,
        stn_state,
        file_ext="csv",
    )